In [1]:
# ── STAGE 4: End-to-End Pipeline ─────────────────────────────────────────────

!pip install -q transformers datasets scikit-learn huggingface_hub

from datasets import load_dataset
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from huggingface_hub import login
import torch, torch.nn as nn, numpy as np, gc

# ── Login ─────────────────────────────────────────────────────────────────────
login()

# ── Data + Preprocessing ─────────────────────────────────────────────────────
dataset  = load_dataset("freococo/burmese-contextual-pragmatics")
train_ds = dataset["train"]

politeness_map = {
    "neutral":"neutral","polite":"polite","informal":"informal",
    "professional":"professional","blunt":"blunt","rude":"rude",
    "very_polite":"polite","friendly":"informal","religious":"polite",
    "impolite":"rude","stern":"blunt","sarcastic":"rude","polite_but_firm":"polite"
}
positive = {"Warm","Sweet","Cheerful","Soft","Sincere","Friendly","Kind",
            "Caring","Gentle","Encouraging","Supportive","Peaceful","Grateful",
            "Romantic","Joyful","Hopeful","Welcoming","Trusting","Approving",
            "Complimentary","Amused","Excited","Moved","Nostalgic","Sympathetic",
            "Pitying","Curious","Playful","Tender","Awe-struck","Awe",
            "Grateful/Humble","Care"}
formal   = {"Professional","Serious","Respectful","Humble","Stern","Firm",
            "Determined","Authoritative","Commanding","Objective","Deep",
            "Intense","Stern/Concerned"}
negative = {"Angry","Annoyed","Hostile","Aggressive","Harsh","Hateful",
            "Disgusted","Cold","Dismissive","Sarcastic","Impolite","Rude",
            "Blunt","Crude","Envious","Dangerous","Fed up","Dry","Indifferent"}
power_map = {
    "equal":"equal","inferior_to_superior":"inferior_to_superior",
    "superior_to_inferior":"superior_to_inferior","any":"any",
    "customer_to_staff":"inferior_to_superior",
    "customer_to_seller":"inferior_to_superior",
    "passenger_to_driver":"inferior_to_superior",
    "patient_to_doctor":"inferior_to_superior",
}

def coarsen_tone(t):
    if t in positive:  return "positive"
    if t in formal:    return "formal"
    if t in negative:  return "negative"
    if t not in {"Neutral","Direct","Objective","Natural","Indifferent"}: return "emotional"
    return "neutral"

def apply_labels(example):
    example["politeness_coarse"] = politeness_map.get(example["politeness"],"neutral")
    example["tone_coarse"]       = coarsen_tone(example["tone"])
    example["power_coarse"]      = power_map.get(example["power_relation"],"equal")
    example["register_label"]    = example["register"]
    return example

train_ds = train_ds.map(apply_labels)

le_pol  = LabelEncoder().fit(train_ds["politeness_coarse"])
le_reg  = LabelEncoder().fit(train_ds["register_label"])
le_pow  = LabelEncoder().fit(train_ds["power_coarse"])
le_tone = LabelEncoder().fit(train_ds["tone_coarse"])

def encode_labels(example):
    example["label"]          = int(le_pol.transform([example["politeness_coarse"]])[0])
    example["label_register"] = int(le_reg.transform([example["register_label"]])[0])
    example["label_power"]    = int(le_pow.transform([example["power_coarse"]])[0])
    example["label_tone"]     = int(le_tone.transform([example["tone_coarse"]])[0])
    return example

train_ds = train_ds.map(encode_labels)

split1     = train_ds.train_test_split(test_size=0.30, seed=42)
split2     = split1["test"].train_test_split(test_size=0.50, seed=42)
test_data  = split2["test"]

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Test set: {len(test_data)} examples")
print("Setup done.")

# ── Step 1: Run Classifier A on test set (predict register) ──────────────────
print("\nLoading Classifier A (register)...")
model_A = AutoModelForSequenceClassification.from_pretrained(
    "annasus10/xlmr-burmese-register"
).eval()

def predict_batch(model, texts, batch_size=32):
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, max_length=MAX_LENGTH, truncation=True,
                          padding=True, return_tensors="pt")
        with torch.no_grad():
            logits = model(**inputs).logits
        all_preds.extend(np.argmax(logits.numpy(), axis=1).tolist())
    return all_preds

texts_A = test_data["utterance"]
preds_A = predict_batch(model_A, texts_A)
pred_registers = [le_reg.classes_[p] for p in preds_A]
print(f"Register predictions done. Sample: {pred_registers[:5]}")

del model_A
gc.collect()
torch.cuda.empty_cache()

# ── Step 2: Run Classifier B on test set (predict power relation) ─────────────
print("\nLoading Classifier B (power relation)...")
model_B = AutoModelForSequenceClassification.from_pretrained(
    "annasus10/xlmr-burmese-power"
).eval()

texts_B = [
    utt + " </s> " + str(ctx).strip()
    for utt, ctx in zip(test_data["utterance"], test_data["context"])
]
preds_B = predict_batch(model_B, texts_B)
pred_powers = [le_pow.classes_[p] for p in preds_B]
print(f"Power predictions done. Sample: {pred_powers[:5]}")

del model_B
gc.collect()
torch.cuda.empty_cache()

# ── Step 3: Run Classifier C on test set (predict tone) ──────────────────────
print("\nLoading Classifier C (tone)...")
model_C = AutoModelForSequenceClassification.from_pretrained(
    "annasus10/xlmr-burmese-tone"
).eval()

texts_C = [
    utt + " </s> " + str(ctx).strip()
    for utt, ctx in zip(test_data["utterance"], test_data["context"])
]
preds_C = predict_batch(model_C, texts_C)
pred_tones = [le_tone.classes_[p] for p in preds_C]
print(f"Tone predictions done. Sample: {pred_tones[:5]}")

del model_C
gc.collect()
torch.cuda.empty_cache()

# ── Step 4: Build Stage 4 inputs using predicted metadata ────────────────────
print("\nBuilding Stage 4 inputs...")
texts_S4 = [
    f"[register: {reg}] [power: {pow}] [tone: {tone}] "
    + utt
    + " </s> " + str(ctx).strip()
    + " </s> " + str(ins).strip()
    for reg, pow, tone, utt, ctx, ins in zip(
        pred_registers, pred_powers, pred_tones,
        test_data["utterance"], test_data["context"], test_data["instruction"]
    )
]
print(f"Sample Stage 4 input:\n  {texts_S4[0]}")

# ── Step 5: Run Stage 3 politeness model on Stage 4 inputs ───────────────────
print("\nLoading Stage 3 politeness model...")
model_S3 = AutoModelForSequenceClassification.from_pretrained(
    "annasus10/xlmr-burmese-pragmatics-stage3-v2"
).eval()

preds_S4 = predict_batch(model_S3, texts_S4)
true_labels = [example["label"] for example in test_data]

print("\n── Stage 4 Results (End-to-End Pipeline) ────────────────────────────")
accuracy  = accuracy_score(true_labels, preds_S4)
macro_f1  = f1_score(true_labels, preds_S4, average="macro")
weighted_f1 = f1_score(true_labels, preds_S4, average="weighted")

print(f"Accuracy:    {accuracy:.4f}")
print(f"Macro F1:    {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")
print()
print(classification_report(true_labels, preds_S4, target_names=le_pol.classes_))

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n── Full Ablation Summary ────────────────────────────────────────────")
print(f"{'Stage':<12} {'Input':<45} {'Macro F1':>10}")
print("-" * 70)
print(f"{'Stage 1':<12} {'Utterance only':<45} {'0.7014':>10}")
print(f"{'Stage 2':<12} {'+ context + instruction':<45} {'0.7583':>10}")
print(f"{'Stage 3':<12} {'+ oracle register/power/tone':<45} {'0.7990':>10}")
print(f"{'Stage 4':<12} {'+ predicted register/power/tone':<45} {macro_f1:>10.4f}")
print("-" * 70)
print(f"\nOracle → Pipeline gap: {0.7990 - macro_f1:.4f} macro F1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

burmese-conversational-pragmatics.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2200 [00:00<?, ? examples/s]

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

...

Tone predictions done. Sample: [np.str_('emotional'), np.str_('neutral'), np.str_('neutral'), np.str_('neutral'), np.str_('emotional')]

Building Stage 4 inputs...
Sample Stage 4 input:
  [register: colloquial] [power: superior_to_inferior] [tone: emotional] နေမကောင်းဖြစ်ရတာ ပင်ပန်းတယ်။ </s> Chronic illness. </s> Tired of being sick.

Loading Stage 3 politeness model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


── Stage 4 Results (End-to-End Pipeline) ────────────────────────────
Accuracy:    0.7273
Macro F1:    0.7020
Weighted F1: 0.7327

              precision    recall  f1-score   support

       blunt       0.89      0.57      0.70        14
    informal       0.72      0.69      0.71        59
     neutral       0.88      0.68      0.77       174
      polite       0.53      0.91      0.67        56
professional       0.83      1.00      0.91        15
        rude       0.43      0.50      0.46        12

    accuracy                           0.73       330
   macro avg       0.71      0.73      0.70       330
weighted avg       0.77      0.73      0.73       330


── Full Ablation Summary ────────────────────────────────────────────
Stage        Input                                           Macro F1
----------------------------------------------------------------------
Stage 1      Utterance only                                    0.7014
Stage 2      + context + instruction       